# Week 4 Lab Assignment

**What is the ask?**  Integrate the quantized XOR NN from this week's programming assignment, removing all traces of floating point from the system.  Evaluate the quantized NN's size, power, and performance requirements vs. the floating-point based NN.  Enhance the design to use exceptions, interrupts, and to change from "keypress" initiated sampling to continuous sampling.

**Why is this important?** Embedded systems are almost always highly optimized designs.  We want to do as much as we can with as "little" as possible.  "little" might be "smallest", "least expensive", "lowest power", but always working against "fast enough."  Knowing how to develop fast, efficient code is a significant differentiator for embedded systems engineers, and demonstrating that your code is fast (or at least fast enough) an essential skill.  One of the ways we write fast, efficient code is to use exceptions and interrupts.  These avoid "busy waiting" with the processing core, and they are the only way to design systems with more than rudimentary behavior.  

NOTE: Debugging exception handlers and interrupt handlers can be quite difficult.  Start early, and leave yourself time to get help if you need it.

NOTE: Most of the effort in this lab is in Part 3.  Do not be mis-lead if you find Parts 1 and 2 "relatively easy."  Part 1 is only relatively easy because you've already done all of the hard stuff.  Part 2 just unlocks the door to the bigger effort in Part 3.  Plan accordingly.

## Project Orientation

This project starts with a clean code repo.  For Part 1: You will copy your NN implementation from last week into this code repo, replacing nn.c with your code (and a few of you also updated nn.h and/or main.c so be sure to bring those changes over too).  For Parts 2 and 3: You will make code changes in main.c, adc.h, and adc.c.  You will use GPIO pin toggling and the Analog Discovery 3's oscilloscope / logic analyzer to characterize system performance. 

## Part 1 - Integrating NN_qpredict()

Replace the contents of nn.c with your quantized implementation code (only!) from this week's programming assignment. Do not bring over any floating point code!  Review the comments in main() to understand how floating point is removed from the raw ADC sample scaling into Qm.n format and how
floating point is removed from output presentation using printf().  

Review all of main.c, nn.h, and nn.c to be sure you have removed all traces of floating point!

## Checkpoint 1: Integrating NN_qpredict()

Build and run your code, and exercise all four possible input values (by *carefully* moving the jumper wires into analog inputs PA0 and PA1 between +3.3 and GND) to verify your implementation.  Your output should resemble mine, below, though your order may be different depending on when you move your jumper wires around and your input values.  Your input values may also be different depending on your fixed-point representation format.

*Replace with a screenshot of your results*

![Part 1 Expected Results](images/Checkpoint1.png)

Grading Rubric [5 points total]:
* -1: for each of 4 possible rows either missing or not showing XOR-like results
* -1: wrong image / missing results in image

The gpio_pin_set() and gpio_pin_reset() calls for Port A Pin #3 are placed immediately before and after the call to NN_qpredict() in main().  Use your oscilloscope to capture and display the execution time of your NN_qpredict().  Be sure to change the time base, range, and X and Y position/offset values to maximize use of the oscilloscope's capture window when capturing the execution time of NN_qpredict().

![Qpredict time](images/Checkpoint2.png)

Grading Rubric: [5 points total]

* -2: Incorrect trigger configuration
* -2: Timebase and X-axis offset does not maximize captured signal display
* -1: Y-axis does not show 0 and +3.3V voltage levels

Complete the table below, referring back to your Week 3 Lab Assignment for the execution time and code/data size of the floating-point based implementation of NN_predict().  Make sure the timing data you copy from Week 3 is only for the execution time of NN_predict() (and not one of the values that includes execution time for ADC_convert() and printf()!)

| Implementation    | Execution Time | flash (text size) | RAM (data + BSS size) |
|-------------------|----------------|-------------------|-----------------------|
| NN_predict()      | 305 us           |   27292            | 840                  |
| NN_**q**predict() | 15.75 us           |   9756            | 480                  |

Grading Rubric: [7 points total]
* -1: for each of the six values that are "wildly" unreasonable
* -1: missing units

How do the execution time, flash, and RAM metrics compare between the two implementations?  Provide a general assessment, but also include statements like "by X percent", or "by a factor of X" or "by x orders of magnitude" for each metric:

I observe that the quantized implementation is more efficient than the floating-point implementation in all aspects. 
For execution time, NN_qpredict() runs about 19× faster than NN_predict() or we could also say 95% reduction in time. 
Flash usage is also very less.  The quantized version uses about 64% less flash memory (9,756 bytes vs. 27,292 bytes).
RAM usage decreases as well. NN_qpredict() uses about 43% less RAM than NN_predict() (480 bytes vs. 840 bytes).

Grading Rubric: [5 points total]
* -1: for any invalid comparison of execution time, flash, RAM
* -2: for an invalid general assessment

But we're not done yet.  Don't forget about compiler optimization! Change the OPT flag in the Makefile from -O0 to -O3 (highest optimization).  What does that do to your flash/RAM utilization and execution time?  (you can copy your results from above for the first row of the table)

| Optimization      | Execution Time | flash (text size) | RAM (data + BSS size) |
|-------------------|----------------|-------------------|-----------------------|
| NN_qpredict() -O0 | 305 us           |   27292            | 840                  |
| NN_qpredict() -O3 | 1.7 us           |   8648            | 480                  |

Grading Rubric: [4 points total]
* -1: for each of the second row values that are "wildly" unreasonable
* -1: missing units 

How do the execution time, flash, and RAM metrics compare between the two optimization levels?  Provide a general assessment, but also include statements like "by X percent", or "by a factor of X" or "by x orders of magnitude" for each metric:

Going from O0 to O3 compiler optimization improves performance and reduces code size by a lot.

With -O3, execution time drops from 305 µs to 1.7 µs. This over 99% reduction in execution time. Flash usage decreases by about 68% (27,292 bytes to 8,648 bytes). RAM usage remains unchanged at 480 bytes.

Grading Rubric: [5 points total]
* -1: for any invalid comparison of execution time, flash, RAM
* -2: for an invalid general assessment

Great stuff!  Be sure to change your optimization level back to -O0 in the Makefile.  Debugging highly optimized code with the debugger can be *bananas*... 

This question is not graded, but I'm curious:  Do you think you should use compiler optimization in this design?  Why/why not?  Does your answer change depending whether you are developing the device, testing/debugging it, or shipping it to a customer?

Yes my answer would depend on the stage.During development and debugging, I would just so do O0 because highly optimized code is harder to debug. During testing and when shipping the product to a customer, I would do -O3 because they significantly improve performance and reduce flash usage.

Back in Lab 1 you refreshed on "power" and how to compute it from current and voltage.  The STM32F042K6 is operating at 3.3V, so use that number for the following question.

Find the "typical current consumption" for the STM32F042K6 the [STM32F042K6 datasheet](datasheets/stm32f042k6.pdf).  The STM32F042K6 is operating in "run" mode, off of a high speed external oscillator (HSE oscillator) in the "bypass" mode with the phased lock loop (PLL) on, at 48 MHz, with (effectively) all peripherals off.  Use this information to compute how much energy (power * time) is required to run a single prediction using NN_predict() and NN_qpredict():
My note: I assume the code is being ran from Flash memory and not RAM. So the current I got was 20.2 mA

| Implementation    | Execution Time | Operating Current | Energy per function call |
|-------------------|----------------|-------------------|-------------------------|
| NN_predict()      | 305 us           |   20.2 mA            | 20.3 uJ                   |
| NN_**q**predict() | 15.75 us           |   20.2 mA            | 1.05 uJ                    |

Grading Rubric: [4 points total]
* -1: for any of operating current, energy for NN_predict, or energy for NN_qpredict that are wrong
* -1: for missing units
  
This question is not graded, but I'm curious:  Assume we can put the STM32F042K6's processing core into an extremely low power "sleep" state when it doesn't have anything interesting to do (yes, we can do that).  Do the above findings suggest the use of floating point vs. fixed-point are better for battery powered designs?  What about for wall-powered designs?  Why/why not?

I think fixed-point is preferred because its faster execution allows the processor to spend more time in low-power sleep. This helps reduce power consumption. For wall-powered designs, power is not that important. So if floating-point makes things easier for development, it can be used. But I think fixed-point still provides better performance and efficiency.

Factoid for you (before answering the last question in this checkpoint): microcontrollers with floating point units "FPUs" (which provide hardware support for floating point) generally cost more than an otherwise identical microcontroller without an FPU.  

Question: Think about the strenghts and weaknesses of fixed point vs. floating point number representations, how much work it takes to convert from float to fixed point, and all of your findings above.  Is there a clear winner?  Or does it still just depend on the design requirements?

 In my opinion there is no absolute winner. The choice depends on the design requirements. Fixed-point arithmetic definitely is more efficient when it comes to power and time and maybe even cost. But floating point is easier to use and debug. It is easier to play around with since you do not need to care about a lot of things. Converting from floating point to fixed point requires a lot of effort too. 

Grading Rubric: [3 points total]
* -1: for not clearly stating your position
* -2: for not providing reasonable justification(s) for your position

When you complete this checkpoint save all files, commit, and sync your work.

## Part 2: Periodic Sampling with SysTick and Exception Handlers

So far you've been pressing a key to initiate collecting ADC samples and running NN predictions.  That's fine, but most sensor-based embedded systems I've worked on *continuously* sample inputs, process data, and either transmit (wired/wireless) a result or take some (control) action... over and over again, until the battery dies or someone unplugs it.  Let's move in that direction.

You've been staring at systick_init() in main(), and systick_callback_function() in main.c - which is magically getting called every ~0.5 seconds to toggle the LED.  It is time to dig in to that magic.  Start by watching the following videos.  As ever, review the SysTick and Exception Handling self assessment in Canvas before you begin to get a sense of where to focus your attention.

[Exceptions Overview](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=5a47151a-0cfa-42b3-834b-b3dc011d2385)

The next video talks specifically about the SysTick timer.  We do not cover hardware timers in any depth in this class, but the video assumes you've seen them.  The basic idea of a hardware timer is that it has an input clock - usually the system clock (ours is 48 MHz) divided down by a scale factor.  The timer counts down from a value, the "reload value", until it gets to zero.  Then it sets a flag (or generates an exception/interrupt!) indicating that it reached zero, reloads its counter from the "reload value register," and starts counting down again. It does all of this without bothering the ARM Cortex M0 processor core, other than to generate an exception (if configured to do so).  We configure the timer by specifying the clock divider and reload value (in the timer's configuration registers).  That's about all SysTick can do, but other "timer" peripherals in the system can do much more (and you can read about this in the [STM32F042K6's Reference Manual](datasheets/rm0091-stm32f0x1stm32f0x2stm32f0x8-advanced-armbased-32bit-mcus-stmicroelectronics.pdf) - look at the Timer sections).  With that in mind, SysTick:

[SysTick Timer](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=89f5d84b-4459-456b-bcb6-b3dc011d23e5)

Having watched the above video, do yourself a favor and trace through main()'s call to systick_init() in the project's systick.c file.  In main(), systick_init() takes a function pointer to the systick_callback_function()... this is setting up the SysTick timer to "call" the systick_callback_function() every time the timer expires.  How does this work?  

Inside systick_init's implementation in systick.c, the function pointer argument is copied to a "static" function pointer called "systick_cbfn".  The "static" keyword, used this way, means the systick_cbfn function pointer is visible to *all code in the file systick.c* (but systick_cbfn is not visible to code outside of the file systick.c).  Afer this, systick_init() sets up SysTick's two primary configuration registers, the reload value register (RVR) and the control and status register (CSR).  Look these up in section 4.4 of the [STM32F042K6 Programming Manaul](datasheets/pm0215-stm32f0-series-cortexm0-programming-manual-stmicroelectronics.pdf) to understand the details of what is happening here.  And that's it.  I said it in the video: SysTick is a very simple hardware timer.

Because systick_init() sets the TICKINT flag in SysTick's CSR (look it up in the Programming Manual, Section 4.4!), every time the timer counts down to zero it will generate a SysTick exception.  You'll see in the next video how the system finds its way to it, but when the SysTick exception is generated, the processing core jumps to the SysTick_Handler() function in systick.c where, if the systick_cbfn function pointer is non-null, the callback function it points to is called.  SysTick_Handler() can call this function because systick_cbfn is "visible to call code in systick.c."  That's how the plumbing works.:

[System Exceptions and Handlers](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=a2bb6147-17e0-4f0b-bab8-b3dc011d2357)

Magic trick revealed!  Take a second to find the g_pfnVectors label in the project's startup_stm32f042x6.s assembly file.  The appropriate entry (based on its offset from the start of the table) has the name of the function to call (e.g. SysTick_Handler). Keep this table in mind.  You'll come back to it later in the lab when you need to find the name of the ADC's *interrupt* handler, but we haven't gotten that far just yet. 

We would expect the SysTick_Handler() to be as well behaved as regular funciton calls and push{} processor registers to the stack before modifying them, then pop{} them back to restore them before returning... and in general it does.  But it takes a little time to "branch" to an exception handler when an exception occurs (the processor has to "look up" the branch address and then, remember that 3-stage processor pipeline and the branch penalty? that takes time too...)  So the ARM Cortex M0 processor core takes advantage of those extra clock cycles to *automatically* do some system state preservation for the exception handler.  This video explains that process, and a few other details, to finish up our coverage of exception handling:

[Managing System State and Interrupts](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=ac67cd28-d84d-43c3-b3f0-b3dc011d240e)

Main takeaways?  Microcontrollers are optimized for high performance exception/interrupt handling, and 'cpsie i' and 'cpsid i' are two important assembly instructions to know.  You either need to know how to mix 'cpsie i' and 'cpsid i' inline in your C code or how to create functions in assembly that you can call from C.  Good news - you know how to do both!

This is a good time to reinforce the previous videos with the SysTick and Exception Handling Self Assessment in Canvas.

## Checkpoint 2 - SysTick and Exception Handlers

This is a fairly quick checkpoint to work through.  But the ideas are more important than the code changes.  While you are in the code, make sure you understand how all the pieces explained in the vidoes and narrative above tie together. Establishing this understanding now will make Interrupts and Interrupt Handling, an expansion on the same ideas involved in exceptions and exception handling, much easier to digest.

Modify the code in main.c and systick.c so the SysTick exception is generated every 100 ms, but modify the systick_callback_function() so that the LED still only blinks at 1 Hz (by toggling its state at 2 Hz).

Add a 16-bit counter variable called systick_events, initialized to zero on system reset, that is incremented inside of the systick_callback_function() every time a SysTick event occurs.

Every time the Systick exception occurs, take a lap through your conditional code block in main() (sample the ADCs, run a prediction, display the results).  

Note that when refactoring code it is common practice to change variable names as necessary if their functionality changes.  I expect this in your code, should the need arise.

Modify the printf() banner in main() to indicate this code does not expect a keypress.  Modify the prediction output in main() to include the number of times the SysTick exception has occurred.  

*Replace my image, below, with a screenshot of your output*

![Expected Results](images/Checkpoint3.png)

Grading Rubric [5 Points Total]:
* -1: printf() banner not updated
* -2: printf() results does not include count: value
* -2: count value does not increment

Copy and paste your systick_init() comments and code showing how you determined, and populated,
the SysTick Reload Value Register (RVR) with the correct value to generate SysTick events at 100ms below:

```
// Implement a 10 Hz SysTick timer event
void systick_init(void(*cbfn)(void)) {
    // register the callback function
    systick_cbfn = cbfn;

    // Trying to configure a 10 Hz SysTick event or 100 ms period.
    // systick uses the external clock source: HCLK / 8.
    //
    // System clock = 48 MHz
    // systick clock = 48 MHz / 8 = 6 MHz
    //
    // wanted systick period = 100 ms = 0.1 seconds
    // timer ticks = 6000000 × 0.1 = 600000
    //
    // The systick RVR stores (N - 1),
    // where N is the number of clock ticks before the counter reaches zero.
    // This is specified in the STM32F042K6 Programming Manual (Section 4.4.2).
    SYSTICK->RVR = 600000-1;
    // Switch to the "external clock source" (HCLK/8 = 6 MHz), 
    // enable the counter,
    // and enable an exception request when the counter reaches 0
    SYSTICK->CSR = (SYSTICK_CSR_ENABLE | SYSTICK_CSR_TICKINT);
}
```

Grading Rubric [3 Points Total]:
* -1: Insufficient Code Commenting
* -1: Incorrect justification for the RVR value 
* -1: Incorrect RVR value

Copy and paste your systick_callback_function() code, as well as any variable definitions for variables used in the systick_callback_function(), as well as any related comments below:

```

// initializing the events variable as instructed to keep count.
// made it volatile because it can be changed in inetrrupt
volatile uint16_t systick_events = 0;
//this just tells main() if a tick happened. 
volatile bool systick_flag = false;

// Callback function for systick exceptions registered in systick_init()
// Toggle the onboard green LED and start an ADC conversion
void systick_callback_function(void) {
    systick_events++;    // increment when an systick event happens
    systick_flag = true; // to let main() know that systick happened, flag is set to false in main()
    // Toggle LED every 5 ticks 
    // systick runs at 10 Hz so LED should toggle at 2 Hz and blink at 1 Hz.
    if ((systick_events % 5) == 0) {
        led_toggle(LED_USER);
    }
}
```

Grading Rubric [10 Points Total]:
* -2: variables missing appropriate 'volatile' and/or 'static' keywords
* -2: variable names not rational for their function
* -2: LED toggling logic and/or led_toggle() calls not in systick_callback_function()
* -2: LED toggling logic to achieve 2 Hz is incorrect
* -2: Insufficient code commenting

When you complete this checkpoint, save all files, commit, and sync your work.

## Part 3: Interrupts, Interrupt Handling, and the ADC

The goal in this section is to switch from a "polling" ADC sampling process to an "interrupt driven" ADC sampling process.  What I mean by the former is in the last few lines of adc_convert() in adc.c:

```
    // Wait for the result
    while( !(ADC->ISR & ADC_ISR_EOC) ); 
    *result = ADC->DR; // clears EOC flag
```

When you request an ADC conversion, by calling adc_convert(), the earlier code in adc_convert() tells the ADC "start a conversion on the requested ADC channel", but then the code (copied above) says "and then sit here doing nothing in a while() loop until the conversion completes."  The gory (but fascinating!) details about how an ADC does the conversion, and how long it takes, can be found in ENGS-28.  The short version is: from a processing core's perspective, ADC conversions take a long time.  We would rather the processor be free to do other useful things (than sit in a while() loop waiting), or we would rather put the processor into a power-saving deep sleep mode while the ADC peforms the requested conversion.  The answer, as you've probably guessed, is to request that the ADC "interrupt" the processor when the conversion completes.  And that is what I will ask you to do later in this assignment.

Okay, so that's the context, and you understand the objective.  Let's extend the idea of exceptions (which occur within the processing core) to the idea of interrupts (which happen outside of the processing core, but want the processing core's attention when they occur).  

The following video introduces the the Nested Vector Interrupt Controller (the NVIC).  The NVIC is a "core peripheral" - it is associated with the processing core and part of the ARM Cortex M0's design.  As such it is described in the [Programming Manual](datasheets/pm0215-stm32f0-series-cortexm0-programming-manual-stmicroelectronics.pdf), section 4.2. The NVIC is responsible for gate-keeping interrupts from all the different peripherals in the system (USARTs, ADCs, Timers, etc) and determining which interrupts, in which order, get the processing core's attention.  

This video uses USART2 as the context for this introduction, and assumes you have been using USART2 with "polling drivers."  That is part of the coursework in ENGG-462, but we skip it in ENGS-62 as polling drivers are ENGS-28's turf.  This project does, however, include an interrupt-driven driver for USART2 so I encourage you to look inside usart.c as you follow along.  usart.c will also serve as a template for you when you enhance adc_init() and adc_convert() to use interrupts later in this assignment.  

[Interrupts and NVIC Overview](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=efc15622-cc49-4f48-87b2-b3dc0132ed75)

The next video switches to the ADC, so it gives you some great (but not complete) support for the task ahead.  usart.c uses the "callback function" approach in its driver, and I'll ask you to do the same with your ADC driver enhancement, so again I suggest looking at usart.c's code as you work through this video:

[Interrupt Handlers](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=fcb7c386-ef2e-4484-9a27-b3dc0132edbc)

The next video gets into a very important concern for any system capable of "concurrent" processing, be it concurrent because there are multiple physical processors in the system or because a single processor is switching between multiple tasks (or interrupt handlers).  When concurrent tasks (or interrupt handlers) share data, we *must* pay attention to data concurrency concerns.  This video sets up the idea, and provides a couple of common methods for addressing some of the more easily resolved situations. 

[Data Concurrency](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=aba58aec-ef2a-4051-ab78-b3dc0132eeab)

When the methods in the previous video are insufficient, I generally reach for "mutexes" next. You do not need to use mutexes this week, but you should be aware of them as we may lean on them in a few weeks.  On our ARM Cortex M0 they are a minoir extension to the idea of "globally disabling and enabling global interrupts," but they minimize the time the system is in an "interrupts disabled for data concurrency reasons" state.  More sophisticated ARM Cortex M-series processor architectures have special machine instructions (see e.g. 'ldrex' and 'strex' assembly instructions in an M4 or M7 ISA document, or internet search) to provide hardware support for mutexes *without disabling interrupts*.  We care about this in the microcontroller world because being responsive to interrupts is a high priority (was that a pun?).  Let's get to the video...

[Mutexes, Deadlock, and Watchdogs](https://dartmouth.hosted.panopto.com/Panopto/Pages/Viewer.aspx?id=24aa2489-bf25-4868-814b-b3dc0132edfd)

Okay, you've seen quite a bit on interrupts, all the problems they solve (responsiveness, efficient processor core use, enabling low power sleep modes).  But they also can create new problems (data concurrency), and solutions to those problems create yet new problems (deadlock).  Things can get complex pretty fast.  Good design decisions help to minimize this complexity, so think about these issues *in advance* of writing code.  That said, let's write some code.

## Checkpoint 3 - Interrupt Driven ADC Driver Enhancements

Paste the following code into main.c below the usart2_rx_callback_handler.  This is what you want the ADC driver to "call" when conversions complete.  

Note the behavior implemented in the switch() {} code block.  We'll start an adc_convert(ADC_CH0) somewhere else in the code, but we want to convert two channels, ADC_CH0 and ADC_CH1, to collect the two input values before we call nn_qpredict().  So in the callback function we check which channel was converted and:
* if CH0, we capture the result and start a conversion on CH1
* if CH1, we capture the result and set a flag to tell main "run the prediction code"

```
// Callback function for ADC end of conversion events
// If the converted channel is channel 0, save the results 
// and start a conversion on channel 1.  If the conversion
// is channel 1, save the results and signal main() to 
// perform a prediction.
static uint16_t ch0; // results of last channel 0 conversion
static uint16_t ch1; // results of last channel 1 conversion
static volatile bool run_prediction = false; // flag for main()
void adc_callback_function(ADC_CHANNEL_t channel, uint16_t data) {
    switch(channel) {
        case ADC_CH0:
            ch0 = data;
            adc_convert(ADC_CH1);
            break;
        case ADC_CH1:
            ch1 = data;
            run_prediction = true;
            break;
        default:
            // Shouldn't ever happen!
    }
}
```

Modify adc_init()'s declaration in adc.h to take a "function pointer to a function taking a an ADC_CHANNEL_t argument and a uint16_t argument and returning void."  You will pass in the adc_callback_function(), above, as the function pointer.

Enhance adc_init()'s implementation in adc.c to:
* "register" this callback function (using a module-level visibility function pointer).  
* Enable interrupts from the ADC peripheral when conversions complete.  
* Enable ADC interrupts in the NVIC so they are passed from the ADC to the ARM Cortex M0 core.

You can find support for these tasks in
* The [STM32F042K6 Reference Manual](datasheets/rm0091-stm32f0x1stm32f0x2stm32f0x8-advanced-armbased-32bit-mcus-stmicroelectronics.pdf) section 13.3.8 and 13.11.X 
* The header files adc.h and nvic.h
* usart.c's implementation as a template for your own

Copy and paste your declaration of the variable you use for the module-level visibility callback functioin pointer (and any associated comments) below:

```

// Module-level pointer to an ADC callback function that is provided
// by the user. This function is called from the ADC interrupt 
// handler when a conversion completes. Set to 0 until registered
// in the adc_init() function below.
static void(*adc_cbfn_ptr)(ADC_CHANNEL_t channel, uint16_t data) = 0;
```

Grading Rubric [2 Points]:
* -1: insufficnet variable declaration (or none)
* -1: insufficient code comments

Copy and paste your implementation of adc_init(), and any associated code comments, below:

```
// enable the ADC and any IO pins used by the ADC using the bus clock (8 Mhz)
// configure for 12-bit sampling mode
void adc_init(void(*adc_cbfn)(ADC_CHANNEL_t channel, uint16_t data)) {

    // Registering the callback function pointer that is passed. 
    adc_cbfn_ptr = adc_cbfn;
    
    // Set the correct sampling time for the input signal's impedance
    ADC->SMPR = ADC_SMPR_13_5;

    // Calibrate the ADC
    ADC->CR |= ADC_CR_ADCAL;
    // hardware clears the bit when calibration completes
    // if your program hangs and you hit break & are sitting on this 
    // line something is wrong with your request to calibrate... 
    // check the requirements in the user manual!
    while( ADC->CR & ADC_CR_ADCAL);

    // Enable the ADC
    ADC->CR |= ADC_CR_ADEN;

    // Wait for the ADC to become ready
    while( !(ADC->ISR & ADC_ISR_ADRDY) );

    // Enable ADC peripheral's interrupts. 
    ADC->IER |= ADC_IER_EOCIE;
    
    // Enabling the ADC interrupts in NVIC Interrupt set-enable register
    NVIC_ISER |= NVIC_ISER_SETENA_ADC_COMP;

    // looking up these defined address values in the .h files made it 
    // very easy 
} 
```

Grading Rubric [10 Points]:
* -2: improper ADC settings to enable interrupts from ADC
* -2: improper NVIC settings to pass interrupts from ADC to processing core
* -2: improper callback function registration
* -2: incorrect callback function type
* -2: insufficient code comments

Next, update adc_convert() as it no longer needs to poll on the end of conversion flag, wait for conversion to complete, and return a result.  But we do want adc_convert() to "remember" which channel it was asked to convert as we will pass this information in the callback function, so add 

```
static ADC_CHANNEL_t active_channel;
```` 

to adc.c.  Iniside of adc_convert(), copy the 'channel' argument to the 'active_channel' variable before starting a conversion.

Copy and paste your modified adc_convert() code below:

```
// A modular-level static variable that will store the channel that was passed
// in the adc_convert function. 
static ADC_CHANNEL_t active_channel;

// initiate a conversion from the selected channel and wait for the result
ret_val_t adc_convert(ADC_CHANNEL_t channel) {

    // register the channel
    active_channel = channel;
    
    // Make sure the ADC is ready
    if( !(ADC->ISR & ADC_ISR_ADRDY ) )
        return RET_ERROR;

    // Configure the channel selection register 
    ADC->CHSELR = channel;

    // Start the conversion
    ADC->CR |= ADC_CR_ADSTART;

    return RET_SUCCESS;
}
```

Grading Rubric [3 Points]:
* -1: does not remove "wait for result" code
* -1: does not remove "return result" code
* -1: does not copy the requested channel to active_channel

Great!  

Next implement an interrupt handler to handle ADC "end of conversion" events.  Start by determining the expected name of the ADC interrupt handler:

What is the expected name of the ADC interrupt handler, and where (in which source file and what line number) did you find this information?

The expected name of the ADC interrupt handler should be   ADC1_IRQHandler. I looked through the startup_stm32f042x6.s  file and found it at line 197 and line 277. 
Grading Rubric [3 Points]:
* -2: wrong source file & line, or none given
* -1: wrong handler name

Now implement the ADC interrupt handler in adc.c.  The steps you must cover in the handler are:
* Check if the interrupt was initiated by an "End of Conversion" interrupt
* If yes: 
  * Read the converted sample date from the ADC's data register
  * Check if the callback function is valid (not 0)
  * Call the callback function, passing it the converted sample data and the channel it was sampled from
  * Make sure the "End of Conversion" flag is cleared before you exit the interrupt handler (otherwise the processor will just immediately return to it again, and again, and again...)

Okay, it is time to test (and debug) your driver!

I suggest starting with the simplest case, a single sample that you manually initiate.  Modify your main.c code to call adc_convert(ADC_CH0) in response to a keypress... you can put this right inside of the usart_rx_callback_function().  Set a breakpoint anywhere inside of your interrupt handler codes (the interupt handler itself in adc.c, the callback function in main).  Do you hit the breakpoint when yo paste a key?  

Debug the simple case first!

When that is working, modify code in main.c so that the run_prediction flag causes code in main() to call nn_qpredict() and display the results.  

Now when you press a key do you get the expected output in your serial terminal emulator?

Almost done!

Let's get back to continuous sampling.  Instead of calling adc_convert(ADC_CH0) in response to key presses, call it every 100ms from the systick_callback_function().

Your output should now resemble your output from the end of Part 2, but you've enhanced the system to use ADC interrupts instead of polling.  Awesome!  

Polling is fine when getting to know a particular peripheral, or for (very) simple systems.  But I can't remember the last time I saw it used in commercial code. 

You're done with the coding part of the (graded) lab.  Let's take a snapshot of the last major code enhancement, then get some measurements and visualize what is going on.  

Copy and paste your interrupt handler code, and any associated comments below:

```
// ADC interrupt handler
void ADC1_IRQHandler(void){
    // declaring a variable to store the converted data 
    uint16_t adc_data;
    // First, check if the interrupt was initiated by EOC interrupt
    if (ADC->ISR & ADC_ISR_EOC){
        // reading the converted data and clearing EOC flag at the same time. 
        adc_data = ADC->DR;

        // if the function pointer we defined at the start is valid, pass the
        // converted data and the channel it was converted from
        if (adc_cbfn_ptr){
            adc_cbfn_ptr(active_channel, adc_data);
        }
    }
}

```

Grading Rubric [10 Points]:
* -2: does not check if interrupt source is end of conversion
* -2: does not check validity of callback function
* -2: does not call callback function with valid sample data or channel
* -2: does not describe how the interrupt flag is cleared
* -2: otherwise insufficient commenting

There are no right or wrong numbers here (I just want to see how much your implementations vary).  Fill out the following table with the build results for your working system:

| text | data | bss |
|------|------|-----|
| 10012 | 100 | 396 |

I do, however, want you to visualize what is going on with the interrupts.  GPIO Port A Pins #3 and #4 are both already enabled as outputs (see rcc_gpio_init() in sysinit.c).  You have been using GPIO Port A Pin #3 to capture code execution time.  Observe what is happening with interrupt behavior in the system by:

* moving gpio_pin_set(GPIOA, gpio_pin_3) to the start of your systick_callback_function()
* moving gpio_pin_reset(GPIOA, gpio_pin_3) right before you exit your systick_callback_function()
* adding gpio_pin_set(GPIOIA, gpio_pin_4) at the start of your adc_callback_function()
* adding gpio_pin_reset(GPIOA, gpio_pin_4) right before you exit your adc_callback_function()
* connecting the AD3's '1+' to pin PA3 (not 'A3' on the [schematic](datasheets/STM32F042K6-Nucleo-Schematics.pdf), 'PA3')!
* connecting the AD3's '2+' to pin PA4 (not 'A4'!)
* connecting the AD3's '1-' and '2-' to your ground rail
* enabling both channels in the WaveForms oscilloscope
* capturing on the rising edge of channel 1, 1 volt, as per usual

Your screen should resemble mine (at least a little bit).  Replace my image below with a snapshot of your own:

![Trace Output](images/Checkpoint4.png)

Grading Rubric [5 Points]:
* -2: Channel 1 results missing / significantly different
* -2: Channel 2 results missing / significantly different
* -1: signal capture does not maximize scope capture window

Finally, interpret your findings:

What does the channel 1 pulse represent and why is it the width it is, relative to the width of the channel 2 pulses?

The channel 1 pulse shows the execution time of the systick_callback_function() interrupt handler. Its width is basically how long the processor spends inside the systick isr executing the code related to a systick event. This means that incrementing the SysTick counter, evaluating the LED toggle condition, toggling the LED and adc_convert(ADC_CH0) all come under this time. Even though we know that adc_convert() doesn't wait for the ADC conversion to complete, it still performs some operations such as updating the active channel, configuring the ADC channel selection register and actually starting the conversion. Because all of this work, the channel 1 pulse is wider than the channel 2 pulses. In the other hand, the channel 2 pulses represent ADC EOC interrupts. This captures the time taken reading the ADC data register, performing a small amount of logic in the ADC callback function, and then exiting. Because the SysTick interrupt performs more instructions rather than responding to completed ones, the channel 1 pulse is wider than the channel 2 pulses.

Grading Rubric [2 Points]:
* -1: does not explain what the pulse represents
* -1: does not rationalize its width relative to channel 2 pulses

What do the channel 2 pulses represent, and why are they different (assuming they are)?

The channel 2 pulses basically captures the time taken by the ADC interrupt handler (ADC1_IRQHandler)that is set off when an EOC interrupt occurs. Each pulse represents one completed ADC conversion. The pulses may differ slightly in width because the ADC callback performs different instructions depending on which channel was converted. When channel 0 completes, the callback stores the result and starts a second conversion on channel 1. This adds extra instructions and increases the execution time. When channel 1 completes, the callback stores the result and sets a flag to signal main() to run the neural network. Then exits without starting another conversion. So the former would take longer. This difference in this work explains why the first channel 2 pulse is slightly wider than the second.

Grading Rubric [2 Points]:
* -1: does not explain what the pulses represent
* -1: does not rationalize their differing widths - if different

The time between the end of the channel 1 pulse and the start of the first channel 2 pulse is the same as the time between the end of the first channel 2 pulse and the start of the second channel 2 pulse (or at least it should be!).  What is happening during these intervals, and why is the duration the same?

In between the intervals between the end of the channel 1 pulse and the start of the first channel 2 pulse as well as between the first and second channel 2 pulses, the ADC hardware performs an analog-to-digital conversion. When adc_convert() starts a conversion, processsor returns to normal execution while the ADC samples and converts the input signal on its own. The duration of these intervals is the same because the ADC conversion time is already fixed. I think it should be determined by the ADC clock. Both channel 0 and channel 1 conversions likely use identical ADC settings. Thus, the hardware requires the same amount of time to complete each conversion which results in equal delays between the pulses.

Grading Rubric [2 Points]:
* -1: capture does not have this timing behavior
* -1: insufficient explanation 


## Bonus Rounds

Looking for more?  Collect a few points (maximum is 100%, however), pick up an extra bit of knowledge, or just gain some bragging rights.  These bits are entirely on you until the assignment is due (no help from me).

### Implement a Finite State Machine 

Use a finite state machine (FSM) in main.c to coordinate between callback functions and main().  There are many ways to implement FSMs - I do not think any one particular approach is better than another.  If you are looking for inspiration, I like to create a new type for state, and create a file-level visible state variable:

```
// Allowed States
typedef enum { state_init, state_1, state_2, ... } state_t;

// State (initialized)
static state_t state = state_init;
```

If an event triggers a state transaction I might code it like this:

```
if( (state == state_wait_for_keypress) && (keypress) ) {
    key_pressed_function(); // perform action
    state = state_next;     // state transition
}
```

Use the state machine to implement:

* Out of reset nothing happens
* When the user presses a key between '1' and '9' the system performs 1-9 predictions at a 100ms sampling interval
* The onboard green LED should illuminate when a key between '1' and '9' is pressed, but it should turn off after sampling is finished.

Let me know that you've implemented this, so I check your code, with a note below:

*Change this text if you implemented the FSM*

### Use an LED to Display Prediction Results

Enable Port A Pin #5 ('PA5' on the schematic) as a digital output.  Instead of printing out prediction results, illuminate an LED if the prediction is above a threshold and turn it off if the prediction is below a threshold.  You can now remove all "printf()" code from your project.  What happens to code size?

| text | data | bss |
|------|------|-----|
|  ?   |    ? |   ? |

(I'll know to check your code if the above values are filled in)

## Conclusion

In this week's lab you integrated the quantized XOR NN from this week's programming assignment, removing all traces of floating point from the system.  You evaluated the quantized NN's size, power, and performance requirements vs. the floating-point based NN and should have seen some remarkable improvements over where we've been the past few weeks.  You also enhance the design to use exceptions, interrupts, and to change from "keypress" initiated sampling to continuous sampling - essential skills to have in your embedded systems engineering toolbox.  Well done!  

When you are happy with your work, and before the assignment is due, be sure to save all files, then commit and sync (push) this project to your GitHub repo.  Unless you make other arrangements with me prior to the assignment deadline, I will grade the last sync (push) to your GitHub repo that occurs prior to the assignment deadline.  

Congratulations on finishing Week #4's lab assignment!